# EDA

In [71]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 25)
pd.set_option('display.max_columns', 20)

data = pd.read_csv('./data/data.csv', delimiter=',')
data.head()

,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,1,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,...,0.716845,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469
1,1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,...,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794
2,1,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,...,0.774670,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474
3,1,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,...,0.739555,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982
4,1,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,...,0.795016,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490


In [44]:
desc = data.describe().T
desc

,count,mean,std,min,25%,50%,75%,max
Bankrupt?,6819.0,3.226280e-02,1.767102e-01,0.0,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
ROA(C) before interest and depreciation before interest,6819.0,5.051796e-01,6.068564e-02,0.0,4.765271e-01,5.027056e-01,5.355628e-01,1.000000e+00
ROA(A) before interest and % after tax,6819.0,5.586249e-01,6.562003e-02,0.0,5.355430e-01,5.598016e-01,5.891572e-01,1.000000e+00
ROA(B) before interest and depreciation after tax,6819.0,5.535887e-01,6.159481e-02,0.0,5.272766e-01,5.522780e-01,5.841051e-01,1.000000e+00
Operating Gross Margin,6819.0,6.079480e-01,1.693381e-02,0.0,6.004447e-01,6.059975e-01,6.139142e-01,1.000000e+00
Realized Sales Gross Margin,6819.0,6.079295e-01,1.691607e-02,0.0,6.004338e-01,6.059759e-01,6.138421e-01,1.000000e+00
Operating Profit Rate,6819.0,9.987551e-01,1.301003e-02,0.0,9.989692e-01,9.990222e-01,9.990945e-01,1.000000e+00
Pre-tax net Interest Rate,6819.0,7.971898e-01,1.286899e-02,0.0,7.973859e-01,7.974636e-01,7.975788e-01,1.000000e+00
After-tax net Interest Rate,6819.0,8.090836e-01,1.360065e-02,0.0,8.093116e-01,8.093752e-01,8.094693e-01,1.000000e+00
Non-industry income and expenditure/revenue,6819.0,3.036229e-01,1.116344e-02,0.0,3.034663e-01,3.035255e-01,3.035852e-01,1.000000e+00


La maggior parte delle variabili hanno minino 0 e massimo 1 -> assumono valori in [0, 1].
Altre invece non sono scalate, procediamo quindi a scalare queste variabili.
Prima di scalarle, applichiamo un /sigma clipping per il rilevamento di eventuali outliers.

In [67]:
#prendiamo le feature che hanno un valore massimo diverso da 1 -> praticamente il subset non scalato
not_scaled_cols = desc[desc['max'] != 1].index
not_scaled_data = data[not_scaled_cols]
not_scaled_data

,Operating Expense Rate,Research and development expense rate,Interest-bearing debt interest rate,Revenue Per Share (Yuan ¥),Total Asset Growth Rate,Net Value Growth Rate,Current Ratio,Quick Ratio,Total debt/Total net worth,Accounts Receivable Turnover,...,Allocation rate per person,Quick Assets/Current Liability,Cash/Current Liability,Inventory/Current Liability,Long-term Liability to Current Assets,Current Asset Turnover Rate,Quick Asset Turnover Rate,Cash Turnover Rate,Fixed Assets to Assets,Total assets to GNP price
0,1.256969e-04,0.000000e+00,7.250725e-04,0.017560,4.980000e+09,0.000327,0.002259,0.001208,0.021266,0.001814,...,0.037135,0.001997,1.473360e-04,0.001036,2.559237e-02,7.010000e+08,6.550000e+09,4.580000e+08,0.424206,0.009219
1,2.897851e-04,0.000000e+00,6.470647e-04,0.021144,6.110000e+09,0.000443,0.006016,0.004039,0.012502,0.001286,...,0.012335,0.004136,1.383910e-03,0.005210,2.394682e-02,1.065198e-04,7.700000e+09,2.490000e+09,0.468828,0.008323
2,2.361297e-04,2.550000e+07,7.900790e-04,0.005944,7.280000e+09,0.000396,0.011543,0.005348,0.021248,0.001495,...,0.141016,0.006302,5.340000e+09,0.013879,3.715116e-03,1.791094e-03,1.022676e-03,7.610000e+08,0.276179,0.040003
3,1.078888e-04,0.000000e+00,4.490449e-04,0.014368,4.880000e+09,0.000382,0.004194,0.002896,0.009572,0.001966,...,0.021320,0.002961,1.010646e-03,0.003540,2.216520e-02,8.140000e+09,6.050000e+09,2.030000e+09,0.559144,0.003252
4,7.890000e+09,0.000000e+00,6.860686e-04,0.029690,5.510000e+09,0.000439,0.006022,0.003727,0.005150,0.001449,...,0.023988,0.004275,6.804636e-04,0.004869,0.000000e+00,6.680000e+09,5.050000e+09,8.240000e+08,0.309555,0.003878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6814,1.510213e-04,4.500000e+09,1.790179e-04,0.020766,7.070000e+09,0.000450,0.010451,0.005457,0.006655,0.000690,...,0.006312,0.005469,5.071548e-03,0.013212,1.792237e-03,2.294154e-04,1.244230e-04,1.077940e-04,0.400338,0.000466
6815,5.220000e+09,1.440000e+09,2.370237e-04,0.023050,5.220000e+09,0.000445,0.009259,0.006741,0.004623,0.000655,...,0.003401,0.006790,4.727181e-03,0.006730,2.204673e-03,1.517299e-04,1.173396e-04,7.710000e+09,0.096136,0.001959
6816,2.509312e-04,1.039086e-04,0.000000e+00,0.044255,5.990000e+09,0.000435,0.038424,0.035112,0.001392,0.001510,...,0.002774,0.035531,8.821248e-02,0.007810,0.000000e+00,1.762272e-04,1.749713e-04,4.074263e-04,0.055509,0.002840
6817,1.236154e-04,2.510000e+09,2.110211e-04,0.031535,7.250000e+09,0.000529,0.012782,0.007256,0.003816,0.000716,...,0.007489,0.007753,7.133218e-03,0.013334,3.200000e+09,2.135940e-04,1.351937e-04,1.165392e-04,0.246805,0.002837


In [73]:
desc_ns = not_scaled_data.describe()
desc_ns

,Operating Expense Rate,Research and development expense rate,Interest-bearing debt interest rate,Revenue Per Share (Yuan ¥),Total Asset Growth Rate,Net Value Growth Rate,Current Ratio,Quick Ratio,Total debt/Total net worth,Accounts Receivable Turnover,...,Allocation rate per person,Quick Assets/Current Liability,Cash/Current Liability,Inventory/Current Liability,Long-term Liability to Current Assets,Current Asset Turnover Rate,Quick Asset Turnover Rate,Cash Turnover Rate,Fixed Assets to Assets,Total assets to GNP price
count,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,...,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03,6.819000e+03
mean,1.995347e+09,1.950427e+09,1.644801e+07,1.328641e+06,5.508097e+09,1.566212e+06,4.032850e+05,8.376595e+06,4.416337e+06,1.278971e+07,...,1.125579e+07,3.592902e+06,3.715999e+07,5.580680e+07,5.416004e+07,1.195856e+09,2.163735e+09,2.471977e+09,1.220121e+06,1.862942e+07
std,3.237684e+09,2.598292e+09,1.082750e+08,5.170709e+07,2.897718e+09,1.141594e+08,3.330216e+07,2.446847e+08,1.684069e+08,2.782598e+08,...,2.945063e+08,1.716209e+08,5.103509e+08,5.820516e+08,5.702706e+08,2.821161e+09,3.374944e+09,2.938623e+09,1.007542e+08,3.764501e+08
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.566874e-04,1.281880e-04,2.030203e-04,1.563138e-02,4.860000e+09,4.409689e-04,7.555047e-03,4.725903e-03,3.007049e-03,7.101336e-04,...,4.120529e-03,5.239776e-03,1.973008e-03,3.163148e-03,0.000000e+00,1.456236e-04,1.417149e-04,2.735337e-04,8.536037e-02,9.036205e-04
50%,2.777589e-04,5.090000e+08,3.210321e-04,2.737571e-02,6.400000e+09,4.619555e-04,1.058717e-02,7.412472e-03,5.546284e-03,9.678107e-04,...,7.844373e-03,7.908898e-03,4.903886e-03,6.497335e-03,1.974619e-03,1.987816e-04,2.247728e-04,1.080000e+09,1.968810e-01,2.085213e-03
75%,4.145000e+09,3.450000e+09,5.325533e-04,4.635722e-02,7.390000e+09,4.993621e-04,1.626953e-02,1.224911e-02,9.273293e-03,1.454759e-03,...,1.502031e-02,1.295091e-02,1.280557e-02,1.114677e-02,9.005946e-03,4.525945e-04,4.900000e+09,4.510000e+09,3.722000e-01,5.269777e-03
max,9.990000e+09,9.980000e+09,9.900000e+08,3.020000e+09,9.990000e+09,9.330000e+09,2.750000e+09,9.230000e+09,9.940000e+09,9.740000e+09,...,9.570000e+09,8.820000e+09,9.650000e+09,9.910000e+09,9.540000e+09,1.000000e+10,1.000000e+10,1.000000e+10,8.320000e+09,9.820000e+09


In [77]:
mean_max = desc_ns.loc[['mean', 'max'], not_scaled_cols]
mean_max.T

,mean,max
Operating Expense Rate,1.995347e+09,9.990000e+09
Research and development expense rate,1.950427e+09,9.980000e+09
Interest-bearing debt interest rate,1.644801e+07,9.900000e+08
Revenue Per Share (Yuan ¥),1.328641e+06,3.020000e+09
Total Asset Growth Rate,5.508097e+09,9.990000e+09
Net Value Growth Rate,1.566212e+06,9.330000e+09
Current Ratio,4.032850e+05,2.750000e+09
Quick Ratio,8.376595e+06,9.230000e+09
Total debt/Total net worth,4.416337e+06,9.940000e+09
Accounts Receivable Turnover,1.278971e+07,9.740000e+09


In [91]:
q1 = desc_ns.loc['25%', not_scaled_cols]
q2 = desc_ns.loc['50%', not_scaled_cols]
q3 = desc_ns.loc['75%', not_scaled_cols]
iqr= q3 - q1
sigma_hat=iqr/1.35

In [93]:
def sigma_clip(df, n_sigma=5):
    df_clean = df.copy()
    
    # Seleziona solo le colonne numeriche
    for col in df_clean.select_dtypes(include=np.number).columns:
        
        # Saltiamo le colonne binarie come 'Bankrupt?' per sicurezza
        if df_clean[col].nunique() <= 2:
            continue
            
        # Rimuoviamo temporaneamente i NaN solo per calcolare i percentili corretti
        valori_validi = df_clean[col].dropna()
        if len(valori_validi) == 0:
            continue # Se la colonna è completamente vuota, passa alla successiva
            
        quartiles = np.percentile(valori_validi, [25, 50, 75])
        median = quartiles[1]
        iqr = quartiles[2] - quartiles[0]
        
        # Se l'IQR è zero, usiamo una stima minima per evitare che sigma_hat sia 0
        if iqr == 0:
            sigma_hat = 1e-6
        else:
            sigma_hat = iqr / 1.349 # 1.349 è più preciso di 1.35
        
        # Calcoliamo i limiti
        lower_limit = median - n_sigma * sigma_hat
        upper_limit = median + n_sigma * sigma_hat
        
        # APPLICHIAMO IL CLIP: Sostituiamo i valori fuori limite senza eliminare le righe!
        df_clean[col] = df_clean[col].clip(lower=lower_limit, upper=upper_limit)
        
    return df_clean

# Adesso puoi lanciarla in totale sicurezza
data_clean = sigma_clip(data)
print(f"Dataset pulito con successo! Righe totali rimaste: {data_clean.shape[0]}")

Dataset pulito con successo! Righe totali rimaste: 6819
